<div style="background:linear-gradient(135deg,#0a2540 0%,#1a3a5c 60%,#0f3460 100%);
            padding:40px 30px;border-radius:12px;text-align:center;margin-bottom:10px;">
  <h1 style="color:#f4a261;font-size:2em;margin:0 0 8px;">
    🔬 Ciencia de Datos en Descubrimiento de Fármacos
  </h1>
  <h2 style="color:#a8dadc;font-size:1.2em;font-weight:400;margin:0 0 16px;">
    07 · Features y espacio químico: Fingerprints · PCA · t-SNE · UMAP
  </h2>
  <p style="color:#cdd6f4;font-size:0.95em;max-width:640px;margin:0 auto;line-height:1.6;">
    Universidad Nacional de Colombia · Extensión UNAL 2026<br>
    <em>Semana 3 — De moléculas a vectores numéricos</em>
  </p>
</div>


---
## ¿Qué es el espacio químico y por qué importa?

Los modelos de machine learning no entienden moléculas — entienden **números**.  
Este notebook responde la pregunta central de la quimioinformática:

> **¿Cómo convertimos una molécula en un vector numérico que capture su información química?**

Y una vez que tenemos vectores, podemos visualizar el **espacio químico**: la distribución
de todas nuestras moléculas en un espacio matemático donde la distancia refleja similitud estructural.

### ¿Qué aprenderás?

| # | Sección | Concepto clave |
|---|---------|---------------|
| 1 | Carga del dataset curado | Conectar con NB-DATA-02 |
| 2 | Descriptores fisicoquímicos | 200+ propiedades calculables con RDKit |
| 3 | Fingerprints moleculares | Morgan (ECFP), MACCS, RDKit FP |
| 4 | Similitud de Tanimoto | Distancia entre moléculas |
| 5 | Correlación y selección de features | Eliminar redundancia |
| 6 | PCA — Análisis de Componentes Principales | Reducción lineal de dimensiones |
| 7 | t-SNE — Visualización no lineal | Preservar estructura local |
| 8 | UMAP — Visualización moderna | Más rápido y escalable que t-SNE |
| 9 | Análisis del espacio químico | ¿Qué regiones cubren los activos? |
| 10 | Diversidad molecular y clustering | Scaffold analysis, agrupamiento |
| 11 | Guardar features para modelado | Matriz lista para Semana 4 |

---
> **Entrada:** `dataset_<target>_curado.csv` (salida de NB-DATA-02)  
> **Salida:** `features_<target>.csv` — matriz de features lista para entrenar modelos


---
## 1. Instalación y carga del dataset curado

In [ ]:
# ── Instalación de librerías ─────────────────────────────────────────────────
#
# ⚠️  INSTRUCCIÓN IMPORTANTE — LEE ANTES DE EJECUTAR:
#
#   1. Ejecuta esta celda → el kernel se reinicia automáticamente
#   2. Ejecuta esta celda DE NUEVO → ahora continúa sin reiniciar
#   3. Continúa con el resto del notebook normalmente
#
# ─────────────────────────────────────────────────────────────────────────────

import subprocess, sys

_paquetes = [
    "numpy>=2.0",
    "pyarrow",
    "rdkit",
    "scikit-learn",
    "umap-learn",
    "matplotlib",
    "seaborn",
    # mordredcommunity es el fork mantenido de mordred compatible con numpy 2.x
    # El mordred original usa np.float (eliminado en numpy 1.24+) y np.bool
    # mordredcommunity corrige esos alias deprecados
    "mordredcommunity>=2.0",
    "tqdm",
]

print("Instalando librerías...")
subprocess.run(
    [sys.executable, "-m", "pip", "install"] + _paquetes + ["--quiet"],
    check=True
)
print("Instalación completada.")
print()

import importlib.metadata
import numpy as _np_mem

_version_memoria = _np_mem.__version__
_version_disco   = importlib.metadata.version("numpy")

print(f"numpy en memoria: {_version_memoria}")
print(f"numpy en disco:   {_version_disco}")

if _version_memoria != _version_disco:
    print()
    print("⚡ Versiones distintas — reiniciando el kernel...")
    print("   → Cuando veas 'sesión reiniciada', ejecuta esta celda de nuevo.")
    from google.colab import runtime
    runtime.unassign()
else:
    import pandas, sklearn, rdkit, pyarrow, umap
    # mordredcommunity se importa igual que mordred
    from mordred import Calculator, descriptors as mordred_desc
    print()
    print(f"✅ numpy              {_version_disco}")
    print(f"✅ pandas             {pandas.__version__}")
    print(f"✅ rdkit              {rdkit.__version__}")
    print(f"✅ sklearn            {sklearn.__version__}")
    print(f"✅ pyarrow            {pyarrow.__version__}")
    print(f"✅ umap               {umap.__version__}")
    print(f"✅ mordredcommunity   {importlib.metadata.version('mordredcommunity')}")
    print()
    print("✅ Todo listo — puedes continuar con el notebook.")
    print()
    print("💡 mordredcommunity se importa exactamente igual que mordred:")
    print("   from mordred import Calculator, descriptors")


In [ ]:
# ── Importaciones ───────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# RDKit — quimioinformática
from rdkit import Chem, DataStructs
from rdkit.Chem import Descriptors, AllChem, Draw, MACCSkeys
from rdkit.Chem.rdMolDescriptors import GetMorganFingerprintAsBitVect
from rdkit.ML.Descriptors import MoleculeDescriptors
from rdkit.Chem.Scaffolds import MurckoScaffold

# Machine learning y reducción de dimensiones
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.feature_selection import VarianceThreshold
from sklearn.cluster import KMeans

# UMAP
try:
    import umap
    UMAP_DISPONIBLE = True
    print("✅ UMAP disponible")
except ImportError:
    UMAP_DISPONIBLE = False
    print("⚠️  UMAP no disponible — instala con: pip install umap-learn")

pd.set_option('display.max_columns', 20)
print("✅ Todo listo")


In [ ]:
# ── Cargar dataset curado desde el repositorio del curso ────────────────────
# Salida del NB-DATA-02 (curación molecular) — EGFR (CHEMBL203)

URL_DATASET = (
    "https://raw.githubusercontent.com/FelPVic/curso_datascience/"
    "d66c544e5f95ace2227e33b6ce833b1aaae1fec7/files/dataset_egfr_curado.csv"
)

print("Cargando dataset curado desde el repositorio del curso...")
df = pd.read_csv(URL_DATASET)

# Normalizar nombres de columnas frecuentes
if 'canonical_smiles' in df.columns and 'std_smiles' not in df.columns:
    df = df.rename(columns={'canonical_smiles': 'std_smiles'})
if 'pValue' in df.columns and 'pActividad' not in df.columns:
    df = df.rename(columns={'pValue': 'pActividad'})

# Verificar estructura mínima necesaria
cols_requeridas = ['std_smiles']
for col in cols_requeridas:
    if col not in df.columns:
        raise ValueError(f"Columna '{col}' no encontrada. Columnas disponibles: {df.columns.tolist()}")

# Filtrar SMILES inválidos
mols_check = [Chem.MolFromSmiles(s) for s in df['std_smiles']]
df = df[[m is not None for m in mols_check]].reset_index(drop=True)

TARGET_NAME = "EGFR (CHEMBL203)"

print(f"✅ Dataset cargado: {len(df)} moléculas")
if 'activo' in df.columns:
    n_act = df['activo'].sum()
    n_ina = (df['activo']==0).sum()
    print(f"   Activos:   {n_act} ({n_act/len(df)*100:.1f}%)")
    print(f"   Inactivos: {n_ina} ({n_ina/len(df)*100:.1f}%)")
if 'pActividad' in df.columns:
    print(f"   pActividad — media: {df['pActividad'].mean():.2f}  "
          f"rango: [{df['pActividad'].min():.1f}, {df['pActividad'].max():.1f}]")
print()
df.head(3)


In [ ]:
# ── Nota: descriptores Mordred ───────────────────────────────────────────────
# Los descriptores Mordred se calculan en la sección 2b de este notebook.
# Si ya los tienes calculados y guardados en parquet, puedes cargarlos así:
#
# import pyarrow.parquet as pq
# df_mordred_final = pd.read_parquet('mordred_egfr.parquet')
# print(f"Mordred cargado: {df_mordred_final.shape}")
#
# Si no, continúa con la sección 2b para calcularlos desde cero.
print("💡 Los descriptores Mordred se calculan en la sección 2b.")


---
## 2. Descriptores fisicoquímicos

Los **descriptores moleculares** son propiedades numéricas calculadas directamente  
desde la estructura 2D o 3D de la molécula. RDKit puede calcular más de 200.

Hay tres categorías principales:

| Categoría | Ejemplos | ¿Qué capturan? |
|-----------|---------|----------------|
| **Constitucionales** | MW, nÁtomos, nAnillos | Tamaño y composición |
| **Electrónicos** | LogP, TPSA, carga | Solubilidad, permeabilidad |
| **Topológicos** | Chi, Kappa, BertzCT | Forma y ramificación |


In [ ]:
# ── ¿Cuántos descriptores puede calcular RDKit? ─────────────────────────────
nombres_descriptores = [d[0] for d in Descriptors.descList]
print(f"Total de descriptores disponibles en RDKit: {len(nombres_descriptores)}")
print()
print("Primeros 20:")
for i, nombre in enumerate(nombres_descriptores[:20]):
    print(f"  {i+1:>3}. {nombre}")
print("  ...")


In [ ]:
# ── Selección de descriptores más relevantes para drug discovery ────────────
# Usamos un subconjunto curado — los 200 descriptores completos tienen
# muchos redundantes y difíciles de interpretar

DESCRIPTORES_SELECCIONADOS = [
    # Constitucionales (tamaño y composición)
    'MolWt', 'HeavyAtomCount', 'NumHeteroatoms',
    'RingCount', 'NumAromaticRings', 'NumAliphaticRings',
    'NumSaturatedRings', 'NumAromaticCarbocycles',

    # Lipinski / ADME (solubilidad, permeabilidad, absorción)
    'MolLogP', 'TPSA', 'NumHDonors', 'NumHAcceptors',
    'NumRotatableBonds', 'MolMR',

    # Electrónicos
    'MaxPartialCharge', 'MinPartialCharge',
    'MaxAbsPartialCharge', 'MinAbsPartialCharge',

    # Topológicos (forma de la molécula)
    'BalabanJ', 'BertzCT', 'Chi0', 'Chi1', 'Chi2v',
    'Kappa1', 'Kappa2', 'Kappa3',

    # Basados en fragmentos
    'fr_amide', 'fr_amine', 'fr_ArN', 'fr_halogen',
    'fr_NH0', 'fr_NH1', 'fr_NH2', 'fr_Ar_OH',
    'fr_C_O', 'fr_nitrile', 'fr_sulfonamd',

    # VSA (van der Waals Surface Area — relacionado con ADME)
    'SlogP_VSA1', 'SlogP_VSA2', 'SlogP_VSA3',
    'SMR_VSA1', 'SMR_VSA2', 'SMR_VSA3',
    'PEOE_VSA1', 'PEOE_VSA2', 'PEOE_VSA3',

    # QED (drug-likeness integrado)
    'qed',
]

# Filtrar solo los disponibles en RDKit
nombres_disponibles = [d[0] for d in Descriptors.descList]
DESC_USAR = [d for d in DESCRIPTORES_SELECCIONADOS if d in nombres_disponibles]

# qed es un módulo aparte
from rdkit.Chem import QED

print(f"Descriptores seleccionados: {len(DESCRIPTORES_SELECCIONADOS)}")
print(f"Disponibles en RDKit:       {len(DESC_USAR)}")


In [ ]:
# ── Calcular descriptores para todo el dataset ───────────────────────────────
from tqdm.auto import tqdm

calc = MoleculeDescriptors.MolecularDescriptorCalculator(DESC_USAR)

def calcular_desc_fila(smiles):
    """Calcula todos los descriptores para un SMILES. Retorna dict o None."""
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return {d: np.nan for d in DESC_USAR + ['qed']}
    try:
        vals = dict(zip(DESC_USAR, calc.CalcDescriptors(mol)))
        vals['qed'] = QED.qed(mol)
        return vals
    except Exception:
        return {d: np.nan for d in DESC_USAR + ['qed']}

print(f"Calculando descriptores para {len(df)} moléculas...")
descriptores_lista = [calcular_desc_fila(s) for s in tqdm(df['std_smiles'])]
df_desc = pd.DataFrame(descriptores_lista)

print(f"\n✅ Matriz de descriptores: {df_desc.shape[0]} moléculas × {df_desc.shape[1]} descriptores")
print(f"   Valores NaN: {df_desc.isna().sum().sum()}")


---
## 2b. Descriptores con Mordred — 1826 descriptores en una línea

**Mordred** (Moriwaki et al. 2018) calcula 1826 descriptores 2D y 3D directamente
desde SMILES. Complementa los descriptores manuales de RDKit con módulos como:
Autocorrelation (606), EState (316), MoRSE (160), RingCount (138), entre otros.

> 💡 Usamos `ignore_3D=True` para calcular solo los 1613 descriptores 2D
> (los 3D requieren un conformero previo). Esto es suficiente para QSAR 2D.


In [ ]:
# ── Instalar e importar Mordred ─────────────────────────────────────────────
try:
    from mordred import Calculator, descriptors as mordred_desc
    MORDRED_OK = True
except ImportError:
    !pip install mordred --quiet
    from mordred import Calculator, descriptors as mordred_desc
    MORDRED_OK = True

print(f"✅ Mordred disponible")

# ── Crear la calculadora — todos los descriptores 2D ─────────────────────────
calc_mordred = Calculator(mordred_desc, ignore_3D=True)

# Ver cuántos descriptores hay disponibles
n_desc_mordred = len(calc_mordred.descriptors)
print(f"   Descriptores 2D disponibles: {n_desc_mordred}")


In [ ]:
# ── Calcular descriptores Mordred para todo el dataset ──────────────────────
from tqdm.auto import tqdm

print(f"Calculando {n_desc_mordred} descriptores Mordred para {len(df)} moléculas...")
print("(puede tardar 25–50 minutos según el tamaño del dataset)")
print()

mols_list = [Chem.MolFromSmiles(s) for s in df['std_smiles']]

# calc_mordred.pandas() devuelve un DataFrame — filas=moléculas, cols=descriptores
df_mordred_raw = calc_mordred.pandas(mols_list)

print(f"✅ Cálculo completado: {df_mordred_raw.shape[0]} × {df_mordred_raw.shape[1]}")
print()
print("Primeros 5 descriptores calculados:")
print(df_mordred_raw.iloc[:3, :5].to_string())


In [ ]:
# ── Limpiar la matriz Mordred ────────────────────────────────────────────────
# Mordred puede devolver objetos de error en lugar de números para algunas moléculas

# Convertir todo a numérico (los errores de Mordred quedan como NaN)
df_mordred_num = df_mordred_raw.apply(pd.to_numeric, errors='coerce')

# Unir con el ID y el Standard smiles para almacenarlo
df_mordred_final = pd.concat([df[['molecule_chembl_id', 'std_smiles']], df_mordred_num], axis=1)
df_mordred_final


In [ ]:
# Guardar el dataset como un parquet para reducir su peso
# Para eso en ves de usar "to_csv" usamos "to_parquet"
# al ser un archivo binario especializado reduce su peso
df_mordred_final.to_parquet('mordred_egfr.parquet', index=False, compression='zstd',
                             compression_level=9)
print("✅ Dataset guardado en mordred_egfr.parquet")

In [ ]:
URL_MORDRED = (
    "https://raw.githubusercontent.com/FelPVic/curso_datascience/"
    "main/files/mordred_egfr.parquet"
)

df_mordred = pd.read_parquet(URL_MORDRED)

In [ ]:
# ── Propiedades fisicoquímicas clave calculadas por Mordred ──────────────────
# Mordred recalcula muchas propiedades de RDKit — aquí mostramos las más relevantes
# para drug discovery

props_fisicoquimicas = {
    'MW':     'Peso molecular (Da)',
    'nHBDon': 'Donadores de puente de H (HBD)',
    'nHBAcc': 'Aceptores de puente de H (HBA)',
    'LogP':   'Lipofilia (LogP)',
    'TPSA':   'Área polar de superficie topológica (Å²)',
    'nRot':   'Número de enlaces rotables',
    'nRing':  'Número de anillos',
    'nArom':  'Número de anillos aromáticos',
}

print("PROPIEDADES FISICOQUÍMICAS (Mordred) — ACTIVOS vs INACTIVOS")
print("=" * 62)
print(f"{'Descriptor':<10} {'Descripción':<35} {'Activos':>8} {'Inactivos':>10}")
print("-" * 62)

for prop, desc in props_fisicoquimicas.items():
    if prop not in df_mordred_final.columns:
        # Intentar variante con mayúsculas/minúsculas
        col_match = [c for c in df_mordred_final.columns
                     if c.lower() == prop.lower()]
        if not col_match:
            continue
        prop = col_match[0]

    vals = df_mordred_final[prop]

    if 'activo' in df.columns:
        mask_act = df['activo'].values[:len(vals)] == 1
        mask_ina = df['activo'].values[:len(vals)] == 0
        med_act = vals[mask_act].median()
        med_ina = vals[mask_ina].median()
        print(f"  {prop:<10} {desc:<35} {med_act:>7.2f}   {med_ina:>8.2f}")
    else:
        print(f"  {prop:<10} {desc:<35} {vals.median():>7.2f}")

print()
print("Valores = mediana por grupo")


In [ ]:
# ── Distribución de propiedades fisicoquímicas: activos vs inactivos ─────────
props_plot = []
for p in ['MW', 'LogP', 'TPSA', 'nHBDon', 'nHBAcc', 'nRot']:
    matches = [c for c in df_mordred_final.columns if c.lower() == p.lower()]
    if matches:
        props_plot.append(matches[0])

if props_plot:
    fig, axes = plt.subplots(2, 2, figsize=(13, 7))
    axes = axes.flatten()

    for ax, prop in zip(axes, props_plot):
        if 'activo' in df.columns:
            mask_act = df['activo'].values[:len(df_mordred_final)] == 1
            mask_ina = df['activo'].values[:len(df_mordred_final)] == 0
            ax.hist(df_mordred_final.loc[mask_ina, prop].dropna(),
                    bins=40, color='#e74c3c', alpha=0.5, label='Inactivo', density=True)
            ax.hist(df_mordred_final.loc[mask_act, prop].dropna(),
                    bins=40, color='#27ae60', alpha=0.65, label='Activo', density=True)
            ax.legend(fontsize=8)
        else:
            ax.hist(df_mordred_final[prop].dropna(), bins=40,
                    color='#7c3aed', alpha=0.8)
        ax.set_xlabel(prop, fontsize=10)
        ax.set_ylabel('Densidad', fontsize=9)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)

    plt.suptitle(f'Propiedades fisicoquímicas (Mordred) — {TARGET_NAME}',
                 fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.savefig('mordred_props_fisicoquimicas.png', dpi=130, bbox_inches='tight')
    plt.show()
    print("✅ Gráfico guardado: mordred_props_fisicoquimicas.png")


In [ ]:
# ── Estadísticas de los descriptores más importantes ────────────────────────
desc_importantes = ['MolWt','MolLogP','TPSA','NumHDonors',
                    'NumHAcceptors','NumRotatableBonds','RingCount','qed']
desc_disp = [d for d in desc_importantes if d in df_desc.columns]

print("ESTADÍSTICAS DE DESCRIPTORES CLAVE")
print("=" * 65)
print(df_desc[desc_disp].describe().round(3).to_string())


---
## 3. Fingerprints moleculares

Los **fingerprints** son vectores binarios (o de conteo) que codifican la presencia  
de subestructuras o fragmentos en la molécula. Son la representación más usada  
en modelos QSAR y búsquedas de similitud.

| Fingerprint | Longitud | ¿Qué codifica? | Mejor para |
|-------------|---------|----------------|------------|
| **Morgan (ECFP4)** | 1024–2048 bits | Entorno circular de cada átomo (radio 2) | Similitud, ML |
| **Morgan (ECFP6)** | 1024–2048 bits | Entorno circular más grande (radio 3) | Mayor detalle estructural |
| **MACCS** | 166 bits | 166 claves estructurales predefinidas | Interpretabilidad |
| **RDKit FP** | 2048 bits | Caminos moleculares (path-based) | Genérico |


In [ ]:
# ── Función para calcular fingerprints ─────────────────────────────────────
def morgan_fp(smiles, radio=2, n_bits=2048):
    """
    Calcula el Morgan Fingerprint (ECFP) como vector numpy binario.

    Parámetros
    ----------
    smiles : str   — SMILES de la molécula
    radio  : int   — radio del entorno (2=ECFP4, 3=ECFP6)
    n_bits : int   — longitud del vector (1024 o 2048)

    Retorna
    -------
    np.array de 0s y 1s, o None si el SMILES es inválido
    """
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    fp = GetMorganFingerprintAsBitVect(mol, radius=radio, nBits=n_bits)
    arr = np.zeros(n_bits, dtype=np.uint8)
    DataStructs.ConvertToNumpyArray(fp, arr)
    return arr

def maccs_fp(smiles):
    """
    Calcula las 166 claves MACCS como vector numpy binario.
    Cada bit corresponde a una clave estructural predefinida.
    """
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    fp = MACCSkeys.GenMACCSKeys(mol)
    arr = np.zeros(167, dtype=np.uint8)
    DataStructs.ConvertToNumpyArray(fp, arr)
    return arr[1:]   # La clave 0 siempre es 0 — la eliminamos

def rdkit_fp(smiles, n_bits=2048):
    """
    Calcula el RDKit Fingerprint (path-based) como vector numpy.
    Codifica todos los caminos de hasta 7 átomos en la molécula.
    """
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    fp = Chem.RDKFingerprint(mol, fpSize=n_bits)
    arr = np.zeros(n_bits, dtype=np.uint8)
    DataStructs.ConvertToNumpyArray(fp, arr)
    return arr

print("Funciones de fingerprints definidas:")
print("  morgan_fp(smiles, radio=2, n_bits=2048)  → ECFP4 (2048 bits)")
print("  morgan_fp(smiles, radio=3, n_bits=2048)  → ECFP6 (2048 bits)")
print("  maccs_fp(smiles)                         → MACCS (166 bits)")
print("  rdkit_fp(smiles, n_bits=2048)            → RDKit FP (2048 bits)")
print()

# Ejemplo con una molécula conocida
ejemplo_smi = df['std_smiles'].iloc[0]
fp_ejemplo = morgan_fp(ejemplo_smi)
print(f"Ejemplo — Molécula: {ejemplo_smi[:50]}...")
print(f"  Morgan FP (ECFP4):  {fp_ejemplo.shape[0]} bits, {fp_ejemplo.sum()} encendidos ({fp_ejemplo.mean()*100:.1f}% densidad)")
fp_maccs = maccs_fp(ejemplo_smi)
print(f"  MACCS:              {fp_maccs.shape[0]} bits, {fp_maccs.sum()} encendidos ({fp_maccs.mean()*100:.1f}% densidad)")


In [ ]:
# ── Calcular Morgan ECFP4 para todo el dataset ───────────────────────────────
print("Calculando Morgan Fingerprints (ECFP4, 2048 bits)...")
fps_morgan = [morgan_fp(s, radio=2, n_bits=2048) for s in tqdm(df['std_smiles'])]

# Filtrar filas con fingerprint válido
mask_fp = [fp is not None for fp in fps_morgan]
df_fp   = df[mask_fp].reset_index(drop=True)
X_morgan = np.array([fp for fp in fps_morgan if fp is not None])

print(f"\n✅ Matriz Morgan ECFP4: {X_morgan.shape[0]} moléculas × {X_morgan.shape[1]} bits")
print(f"   Densidad media: {X_morgan.mean()*100:.2f}% de bits encendidos")


In [ ]:
# ── Calcular MACCS keys ─────────────────────────────────────────────────────
print("Calculando MACCS Keys (166 bits)...")
fps_maccs = [maccs_fp(s) for s in tqdm(df['std_smiles'])]

mask_maccs = [fp is not None for fp in fps_maccs]
X_maccs = np.array([fp for fp in fps_maccs if fp is not None])

print(f"\n✅ Matriz MACCS: {X_maccs.shape[0]} moléculas × {X_maccs.shape[1]} bits")
print()

# MACCS interpretables: mostrar las claves más frecuentes
nombres_maccs = [
    'Key1','Key2','Key3','Key4','Key5','Key6','Key7','Key8','Key9','Key10',
    'ISOTOPE','ATOMIC_NUM_103','ATOMIC_NUM_102','ATOMIC_NUM_101','G1','G2',
    'Any_Ring','Het_Ring','Het_Ring2','Ring_Jnct','Ring_Sz_3','Ring_Sz_4',
    'Ring_Sz_5','Ring_Sz_6','Ring_Sz_7','Ring_Sz_8','Ring_Sz_9','Ring_Sz_10',
    'Arom_Ring','HBDon','HBAcc','NO_Fragment','Halide','Aldehyde','Thioester',
    'Sulfonic_Acid','Phosphoric_Acid','Mono_Halo','Disulfide','Acyl_Halide',
]  # nombres parciales para ilustración

frecuencias = X_maccs.mean(axis=0)
top_idx = np.argsort(frecuencias)[::-1][:10]
print("Claves MACCS más frecuentes en el dataset:")
print(f"  {'Clave':<8} {'Frecuencia':>12}")
for idx in top_idx:
    print(f"  MACCS-{idx+1:<4} {frecuencias[idx]*100:>10.1f}%")


---
## 4. Similitud de Tanimoto entre moléculas

La **similitud de Tanimoto** (o coeficiente de Jaccard) mide cuánto se parecen  
dos fingerprints binarios. Va de 0 (nada similares) a 1 (idénticos).

$$T(A, B) = \frac{|A \cap B|}{|A \cup B|} = \frac{c}{a + b - c}$$

Donde $a$ y $b$ son los bits encendidos en A y B, y $c$ los bits comunes.

> **Regla práctica:** $T \geq 0.85$ → muy similares · $T \geq 0.4$ → similitud moderada


In [ ]:
# ── Similitud de Tanimoto: función y ejemplos ────────────────────────────────
def tanimoto(fp1, fp2):
    """Calcula la similitud de Tanimoto entre dos vectores binarios numpy."""
    interseccion = np.logical_and(fp1, fp2).sum()
    union        = np.logical_or(fp1, fp2).sum()
    if union == 0:
        return 0.0
    return float(interseccion) / float(union)

# Comparar pares de moléculas del dataset
print("EJEMPLOS DE SIMILITUD DE TANIMOTO (Morgan ECFP4)")
print("=" * 60)
indices = [0, 1, 2, 5, 10]
for i in range(len(indices)-1):
    for j in range(i+1, len(indices)):
        a, b = indices[i], indices[j]
        t = tanimoto(X_morgan[a], X_morgan[b])
        smi_a = df_fp['std_smiles'].iloc[a][:30]
        smi_b = df_fp['std_smiles'].iloc[b][:30]
        act_a = df_fp['activo'].iloc[a] if 'activo' in df_fp else '?'
        act_b = df_fp['activo'].iloc[b] if 'activo' in df_fp else '?'
        label_a = '🟢' if act_a==1 else '🔴'
        label_b = '🟢' if act_b==1 else '🔴'
        print(f"  Mol {a} {label_a} vs Mol {b} {label_b}: T = {t:.3f}")


In [ ]:
# ── Distribución de similitudes (Tanimoto pairwise) ─────────────────────────
# Calcular una muestra de similitudes por pares (no todas — serían n²)
np.random.seed(42)
n_muestra = min(500, len(X_morgan))
idx_muestra = np.random.choice(len(X_morgan), n_muestra, replace=False)
X_muestra = X_morgan[idx_muestra]

similitudes = []
for i in range(len(X_muestra)):
    for j in range(i+1, min(i+50, len(X_muestra))):
        similitudes.append(tanimoto(X_muestra[i], X_muestra[j]))

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(similitudes, bins=50, color='#7c3aed', alpha=0.8,
        edgecolor='white', linewidth=0.4)
ax.axvline(0.4, color='#f39c12', linestyle='--', linewidth=1.5,
           label='T=0.4 (similitud moderada)')
ax.axvline(0.85, color='#e74c3c', linestyle='--', linewidth=1.5,
           label='T=0.85 (muy similares)')
ax.set_xlabel('Similitud de Tanimoto', fontsize=11)
ax.set_ylabel('Frecuencia', fontsize=11)
ax.set_title(f'Distribución de similitudes por pares\n(muestra n={len(similitudes)} pares)', fontsize=11)
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig('distribucion_tanimoto.png', dpi=120, bbox_inches='tight')
plt.show()
media_sim = np.mean(similitudes)
print(f"Similitud media entre pares: {media_sim:.3f}")
if media_sim < 0.3:
    print("→ Dataset diverso (baja similitud media) — bueno para generalización")
elif media_sim > 0.6:
    print("→ Dataset poco diverso (alta similitud media) — riesgo de sobreajuste")


---
## 5. Selección de features sobre descriptores Mordred

Con 1613 descriptores la matriz tiene mucha redundancia. Aplicamos cuatro filtros
en orden, imprimiendo cuántas columnas se eliminan en cada paso:

1. **Columnas no numéricas** — eliminar `molecule_chembl_id` y `std_smiles`
2. **NaN > 20%** — descriptores que fallan en muchas moléculas
3. **Varianza casi cero** — descriptores constantes
4. **Test de normalidad → Spearman o Pearson → correlación alta** — redundancia

> Entrada: `df_mordred_final` (1615 columnas incluyendo ID y SMILES)
> Salida: `df_desc_final` + `X_scaled` listos para PCA, t-SNE y UMAP


In [ ]:
# ── Paso 1: Quedarse solo con columnas numéricas ────────────────────────────
# df_mordred_final tiene: molecule_chembl_id, std_smiles + 1613 descriptores
COLS_META = ['molecule_chembl_id', 'std_smiles']

df_sel = df_mordred_final.drop(
    columns=[c for c in COLS_META if c in df_mordred_final.columns]
)
# Asegurar que todo es numérico (Mordred a veces deja objetos de error)
df_sel = df_sel.apply(pd.to_numeric, errors='coerce')

n0 = df_sel.shape[1]
print(f"Mordred inicial (solo descriptores): {df_mordred_final.shape[0]} × {n0}")


In [ ]:
# ── Paso 2: Eliminar columnas con > 20% NaN ─────────────────────────────────
UMBRAL_NAN = 0.20

nan_frac  = df_sel.isna().mean()
cols_nan  = nan_frac[nan_frac > UMBRAL_NAN].index.tolist()
df_sel    = df_sel.drop(columns=cols_nan)
n1        = df_sel.shape[1]

# Imputar NaN restantes con la mediana de cada columna
df_sel = df_sel.fillna(df_sel.median(numeric_only=True))

print(f"Paso 2 — NaN > {UMBRAL_NAN*100:.0f}%:        -{len(cols_nan):>4} columnas → quedan {n1}")


In [ ]:
# ── Paso 3: Eliminar descriptores con varianza casi cero ────────────────────
from sklearn.feature_selection import VarianceThreshold

selector_var = VarianceThreshold(threshold=0.01)
X_var        = selector_var.fit_transform(df_sel)
cols_var     = df_sel.columns[selector_var.get_support()].tolist()
n_elim_var   = df_sel.shape[1] - len(cols_var)
n2           = len(cols_var)

df_sel = pd.DataFrame(X_var, columns=cols_var)

print(f"Paso 3 — Varianza < 0.01:   -{n_elim_var:>4} columnas → quedan {n2}")


In [ ]:
# ── Paso 4a: Test de normalidad → decidir Pearson o Spearman ─────────────────
# Shapiro-Wilk si n < 5000; D'Agostino si n >= 5000
from scipy import stats as scipy_stats

ALPHA = 0.05
n_normales, n_no_normales = 0, 0

for col in df_sel.columns:
    serie = df_sel[col].dropna()
    if len(serie) < 8:
        n_no_normales += 1
        continue
    try:
        _, p = (scipy_stats.shapiro(serie[:5000]) if len(serie) < 5000
                else scipy_stats.normaltest(serie))
        if p > ALPHA: n_normales += 1
        else:         n_no_normales += 1
    except Exception:
        n_no_normales += 1

total        = n_normales + n_no_normales
pct_no_norm  = n_no_normales / total

print(f"Test de normalidad (α = {ALPHA}):")
print(f"  Normales:     {n_normales:>4} ({n_normales/total*100:.1f}%)")
print(f"  No normales:  {n_no_normales:>4} ({n_no_normales/total*100:.1f}%)")
print()

# Si >= 50% no son normales → Spearman para todos (más conservador)
METODO_CORR = 'spearman' if pct_no_norm >= 0.5 else 'pearson'
print(f"→ Método elegido: {METODO_CORR.upper()}")


In [ ]:
# ── Paso 4b: Eliminar descriptores con correlación alta ─────────────────────
UMBRAL_CORR = 0.92

print(f"Calculando matriz de correlación ({METODO_CORR})...")
corr_matrix = df_sel.corr(method=METODO_CORR).abs()
upper       = corr_matrix.where(
    np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
)

# Para cada par redundante eliminar el de MENOR varianza
varianzas      = df_sel.var()
cols_eliminar  = set()
for col in upper.columns:
    for par in upper.index[upper[col] > UMBRAL_CORR].tolist():
        if col in cols_eliminar or par in cols_eliminar:
            continue
        cols_eliminar.add(
            par if varianzas[col] >= varianzas[par] else col
        )

df_desc_final = df_sel.drop(columns=list(cols_eliminar)).copy()
n_elim_corr   = len(cols_eliminar)
n3            = df_desc_final.shape[1]

print(f"Paso 4 — Correlación > {UMBRAL_CORR}: -{n_elim_corr:>4} columnas → quedan {n3}")
print()
print(f"{'─'*45}")
print(f"  RESUMEN SELECCIÓN DE FEATURES")
print(f"{'─'*45}")
print(f"  Mordred inicial:             {n0:>5}")
print(f"  Tras NaN > {UMBRAL_NAN*100:.0f}%:            {n1:>5}  (-{n0-n1})")
print(f"  Tras varianza < 0.01:        {n2:>5}  (-{n1-n2})")
print(f"  Tras correlación {METODO_CORR[:4]} > {UMBRAL_CORR}: {n3:>5}  (-{n2-n3})")
print(f"  Reducción total:             {(1-n3/n0)*100:.1f}%")


In [ ]:
# ── Paso 5: Escalar con StandardScaler ───────────────────────────────────────
from sklearn.preprocessing import StandardScaler

scaler   = StandardScaler()
X_scaled = scaler.fit_transform(df_desc_final)

print(f"✅ X_scaled listo: {X_scaled.shape[0]} moléculas × {X_scaled.shape[1]} features")
print()
print("Variables disponibles para los siguientes pasos:")
print(f"  df_desc_final  → {df_desc_final.shape[1]} descriptores curados (DataFrame)")
print(f"  X_scaled       → mismo contenido escalado (numpy array)")


In [ ]:
# ── Visualizar la matriz de correlación tras el filtro ───────────────────────
n_plot    = min(25, df_desc_final.shape[1])
cols_plot = df_desc_final.columns[:n_plot]
corr_plot = df_desc_final[cols_plot].corr(method=METODO_CORR)

fig, ax = plt.subplots(figsize=(12, 10))
mask = np.triu(np.ones_like(corr_plot, dtype=bool))
sns.heatmap(
    corr_plot, mask=mask, annot=True, fmt='.2f',
    cmap='RdBu_r', center=0, vmin=-1, vmax=1, ax=ax,
    linewidths=0.4, annot_kws={'size': 7},
    cbar_kws={'label': f'Correlación de {METODO_CORR.capitalize()}'}
)
ax.set_title(
    f'Correlación {METODO_CORR.capitalize()} — descriptores curados '
    f'({df_desc_final.shape[1]} total, primeros {n_plot})',
    fontsize=11, pad=10
)
plt.tight_layout()
plt.savefig('correlacion_mordred_curado.png', dpi=130, bbox_inches='tight')
plt.show()
print("✅ Guardado: correlacion_mordred_curado.png")


---
## 6. PCA — Análisis de Componentes Principales

El **PCA** es una reducción lineal de dimensiones. Encuentra las direcciones  
de máxima varianza en los datos y proyecta las moléculas sobre ellas.

**Limitación:** solo captura relaciones *lineales* entre variables.  
Ideal como primer paso exploratorio y para entender qué descriptores dominan.


In [ ]:
# ── PCA sobre descriptores escalados ────────────────────────────────────────
pca = PCA(n_components=10, random_state=42)
X_pca = pca.fit_transform(X_scaled)

# Varianza explicada
varianza_acumulada = pca.explained_variance_ratio_.cumsum()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Scree plot
ax1 = axes[0]
ax1.bar(range(1, 11), pca.explained_variance_ratio_ * 100,
        color='#7c3aed', alpha=0.8, edgecolor='white')
ax1.plot(range(1, 11), varianza_acumulada * 100,
         'o-', color='#f39c12', linewidth=2, markersize=6, label='Acumulada')
ax1.axhline(80, color='gray', linestyle='--', linewidth=1, alpha=0.6, label='80%')
ax1.set_xlabel('Componente Principal', fontsize=11)
ax1.set_ylabel('Varianza explicada (%)', fontsize=11)
ax1.set_title('Scree plot — varianza por componente', fontsize=11)
ax1.legend(fontsize=9)

# PC1 vs PC2 coloreado por pActividad
ax2 = axes[1]
if 'activo' in df_fp.columns:
    colores_act = {1: '#27ae60', 0: '#e74c3c'}
    for clase, label in [(1,'Activo'), (0,'Inactivo')]:
        mask = df_fp['activo'] == clase
        ax2.scatter(X_pca[mask, 0], X_pca[mask, 1],
                    c=colores_act[clase], label=label,
                    alpha=0.5, s=18, edgecolors='none')
    ax2.legend(fontsize=10)
else:
    sc = ax2.scatter(X_pca[:, 0], X_pca[:, 1],
                     c=df_fp['pActividad'] if 'pActividad' in df_fp.columns else 'steelblue',
                     cmap='RdYlGn', alpha=0.5, s=18)
    plt.colorbar(sc, ax=ax2, label='pActividad')

ax2.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)', fontsize=11)
ax2.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)', fontsize=11)
ax2.set_title('PCA — PC1 vs PC2', fontsize=11)

plt.suptitle('PCA sobre descriptores fisicoquímicos', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('pca_descriptores.png', dpi=120, bbox_inches='tight')
plt.show()

print(f"Varianza explicada PC1+PC2: {varianza_acumulada[1]*100:.1f}%")
print(f"Varianza explicada 5 PCs:   {varianza_acumulada[4]*100:.1f}%")
print(f"Varianza explicada 10 PCs:  {varianza_acumulada[9]*100:.1f}%")


In [ ]:
# ── Loadings: qué descriptores dominan cada componente ─────────────────────
# Los loadings explican qué descriptores tienen más peso en cada PC

loadings = pd.DataFrame(
    pca.components_[:3].T,
    index=df_desc_final.columns,
    columns=['PC1', 'PC2', 'PC3']
)

print("TOP 8 DESCRIPTORES QUE MÁS CONTRIBUYEN A CADA PC")
print("=" * 55)
for pc in ['PC1', 'PC2', 'PC3']:
    top = loadings[pc].abs().sort_values(ascending=False).head(8)
    print(f"\n  {pc}:")
    for desc, val in top.items():
        signo = '+' if loadings.loc[desc, pc] > 0 else '-'
        print(f"    {signo}{val:.3f}  {desc}")

print()
print("💡 Los loadings positivos aumentan el valor del PC,")
print("   los negativos lo disminuyen.")


In [ ]:
# ── PCA sobre fingerprints Morgan ───────────────────────────────────────────
# Los fingerprints son datos de alta dimensión (2048 bits) — PCA los comprime

pca_fp = PCA(n_components=2, random_state=42)
X_pca_fp = pca_fp.fit_transform(X_fp)

fig, ax = plt.subplots(figsize=(8, 6))

if 'activo' in df_fp.columns:
    for clase, label, color in [(1,'Activo','#27ae60'), (0,'Inactivo','#e74c3c')]:
        mask = df_fp['activo'] == clase
        ax.scatter(X_pca_fp[mask, 0], X_pca_fp[mask, 1],
                   c=color, label=label, alpha=0.45, s=16, edgecolors='none')
    ax.legend(fontsize=11, markerscale=2)
else:
    ax.scatter(X_pca_fp[:, 0], X_pca_fp[:, 1], alpha=0.45, s=16)

ax.set_xlabel(f'PC1 ({pca_fp.explained_variance_ratio_[0]*100:.1f}%)', fontsize=11)
ax.set_ylabel(f'PC2 ({pca_fp.explained_variance_ratio_[1]*100:.1f}%)', fontsize=11)
ax.set_title('PCA sobre Morgan Fingerprints (ECFP4) Espacio químico 2D', fontsize=12)
plt.tight_layout()
plt.savefig('pca_fingerprints.png', dpi=120, bbox_inches='tight')
plt.show()

print(f"Varianza explicada (FP): PC1={pca_fp.explained_variance_ratio_[0]*100:.1f}%  "
      f"PC2={pca_fp.explained_variance_ratio_[1]*100:.1f}%")
print()
print("💡 Los fingerprints son muy dispersos (sparse) — PCA captura poco.")
print("   t-SNE y UMAP (secciones 7-8) son mejores para fingerprints.")


---
## 7. t-SNE — Visualización no lineal del espacio químico

El **t-SNE** (t-distributed Stochastic Neighbor Embedding) es un método de reducción  
de dimensiones **no lineal** que preserva la estructura local: moléculas similares  
quedan juntas en el mapa 2D.

**Ventajas:** excelente para visualizar clusters y fronteras entre clases  
**Limitaciones:** no preserva distancias globales; lento para datasets grandes; estocástico


In [ ]:
# ── t-SNE sobre fingerprints Morgan ────────────────────────────────────────
# Nota: t-SNE es lento — para datasets > 5000 muestras, usar submuestra o UMAP

N_MAX_TSNE = 3000  # límite para no tardar demasiado en Colab
if len(X_fp) > N_MAX_TSNE:
    idx_sub = np.random.choice(len(X_fp), N_MAX_TSNE, replace=False)
    X_fp_sub = X_fp[idx_sub]
    df_fp_sub = df_fp.iloc[idx_sub].reset_index(drop=True)
    print(f"Dataset grande — usando submuestra de {N_MAX_TSNE} moléculas para t-SNE")
else:
    X_fp_sub = X_fp
    df_fp_sub = df_fp.reset_index(drop=True)
    print(f"Usando dataset completo ({len(X_fp_sub)} moléculas)")

print("Calculando t-SNE (puede tardar 1–3 minutos)...")
tsne = TSNE(
    n_components=2,
    perplexity=35,          # balance local/global (5–50, default 30)
    learning_rate='auto',
    n_iter=1000,
    random_state=42,
    metric='jaccard',       # métrica apropiada para fingerprints binarios
    init='pca'
)
X_tsne = tsne.fit_transform(X_fp_sub)
print("✅ t-SNE calculado")


In [ ]:
# ── Visualización t-SNE: activos vs inactivos ──────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Panel izquierdo: activo / inactivo
ax1 = axes[0]
if 'activo' in df_fp_sub.columns:
    for clase, label, color, alpha in [(0,'Inactivo','#e74c3c',0.35),
                                        (1,'Activo',  '#27ae60',0.65)]:
        mask = df_fp_sub['activo'] == clase
        ax1.scatter(X_tsne[mask, 0], X_tsne[mask, 1],
                    c=color, label=f'{label} (n={mask.sum()})',
                    alpha=alpha, s=12, edgecolors='none')
    ax1.legend(fontsize=10, markerscale=2)

ax1.set_xlabel('t-SNE 1', fontsize=11)
ax1.set_ylabel('t-SNE 2', fontsize=11)
ax1.set_title('t-SNE — Activo vs Inactivo', fontsize=12)
ax1.set_xticks([]); ax1.set_yticks([])

# Panel derecho: coloreado por pActividad continua
ax2 = axes[1]
col_pact = 'pActividad' if 'pActividad' in df_fp_sub.columns else None
if col_pact:
    sc = ax2.scatter(X_tsne[:, 0], X_tsne[:, 1],
                     c=df_fp_sub[col_pact], cmap='RdYlGn',
                     vmin=df_fp_sub[col_pact].quantile(0.05),
                     vmax=df_fp_sub[col_pact].quantile(0.95),
                     alpha=0.5, s=12, edgecolors='none')
    cbar = plt.colorbar(sc, ax=ax2)
    cbar.set_label('pActividad (-log₁₀[IC50 M])', fontsize=10)
else:
    ax2.scatter(X_tsne[:, 0], X_tsne[:, 1], alpha=0.5, s=12)

ax2.set_xlabel('t-SNE 1', fontsize=11)
ax2.set_ylabel('t-SNE 2', fontsize=11)
ax2.set_title('t-SNE — Gradiente de pActividad', fontsize=12)
ax2.set_xticks([]); ax2.set_yticks([])

plt.suptitle(f't-SNE del espacio químico — Morgan ECFP4\n(n={len(X_fp_sub)}, perplexity=35)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('tsne_espacio_quimico.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ── Efecto del hiperparámetro perplexity ────────────────────────────────────
# La perplexity controla el balance entre estructura local y global
# Valores bajos: más detalle local (clusters pequeños)
# Valores altos: más estructura global

print("💡 GUÍA PARA ELEGIR HIPERPARÁMETROS DE t-SNE")
print("=" * 55)
print()
print("  perplexity:")
print("    5–15  → Estructura muy local, clusters pequeños")
print("    30–50 → Balance recomendado (default=30)")
print("    100+  → Estructura más global, grupos grandes")
print()
print("  n_iter:")
print("    250   → Rápido pero puede no converger")
print("    1000  → Recomendado (default)")
print("    2000+ → Para datasets complejos")
print()
print("  metric para fingerprints binarios:")
print("    'jaccard'   → equivalente a Tanimoto ← recomendado")
print("    'euclidean' → menos apropiado para bits")
print("    'cosine'    → alternativa válida")
print()
print("  init:")
print("    'pca'    → más estable y reproducible ← recomendado")
print("    'random' → puede dar resultados distintos entre ejecuciones")
print()
print("⚠️  Las distancias ABSOLUTAS en t-SNE no tienen significado.")
print("   Solo importa si los puntos están juntos o separados.")


---
## 8. UMAP — Visualización moderna y escalable

**UMAP** (Uniform Manifold Approximation and Projection) es una alternativa  
más reciente y rápida a t-SNE. Preserva mejor la estructura global del espacio  
y es mucho más eficiente computacionalmente.

| | t-SNE | UMAP |
|--|-------|------|
| **Velocidad** | Lento (O(n²)) | Rápido (casi lineal) |
| **Estructura local** | ✅ Excelente | ✅ Buena |
| **Estructura global** | ⚠️ Limitada | ✅ Mejor |
| **Reproducibilidad** | Variable | Controlable con seed |
| **Escalabilidad** | < 10.000 pts | > 100.000 pts |


In [ ]:
# ── UMAP sobre fingerprints Morgan ──────────────────────────────────────────
if UMAP_DISPONIBLE:
    print("Calculando UMAP (debería ser más rápido que t-SNE)...")
    reducer = umap.UMAP(
        n_components=2,
        n_neighbors=15,         # balance local/global (5–50)
        min_dist=0.1,           # distancia mínima entre puntos en el embedding
        metric='jaccard',       # Tanimoto para fingerprints binarios
        random_state=42,
        verbose=False
    )
    X_umap = reducer.fit_transform(X_fp_sub)

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    # Panel izquierdo: activo / inactivo
    ax1 = axes[0]
    if 'activo' in df_fp_sub.columns:
        for clase, label, color, alpha in [(0,'Inactivo','#e74c3c',0.35),
                                            (1,'Activo',  '#27ae60',0.65)]:
            mask = df_fp_sub['activo'] == clase
            ax1.scatter(X_umap[mask, 0], X_umap[mask, 1],
                        c=color, label=f'{label} (n={mask.sum()})',
                        alpha=alpha, s=12, edgecolors='none')
        ax1.legend(fontsize=10, markerscale=2)
    ax1.set_title('UMAP — Activo vs Inactivo', fontsize=12)
    ax1.set_xlabel('UMAP 1', fontsize=11)
    ax1.set_ylabel('UMAP 2', fontsize=11)
    ax1.set_xticks([]); ax1.set_yticks([])

    # Panel derecho: pActividad continua
    ax2 = axes[1]
    if 'pActividad' in df_fp_sub.columns:
        sc = ax2.scatter(X_umap[:, 0], X_umap[:, 1],
                         c=df_fp_sub['pActividad'], cmap='RdYlGn',
                         vmin=df_fp_sub['pActividad'].quantile(0.05),
                         vmax=df_fp_sub['pActividad'].quantile(0.95),
                         alpha=0.5, s=12, edgecolors='none')
        plt.colorbar(sc, ax=ax2, label='pActividad')
    ax2.set_title('UMAP — Gradiente de pActividad', fontsize=12)
    ax2.set_xlabel('UMAP 1', fontsize=11)
    ax2.set_ylabel('UMAP 2', fontsize=11)
    ax2.set_xticks([]); ax2.set_yticks([])

    plt.suptitle(f'UMAP del espacio químico — Morgan ECFP4\n(n={len(X_fp_sub)}, n_neighbors=15)',
                 fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.savefig('umap_espacio_quimico.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("✅ UMAP calculado y visualizado")
else:
    print("⚠️  UMAP no disponible. Instala con: !pip install umap-learn")
    print("   Reejecutar esta celda después de instalar.")


In [ ]:
# ── Guía de hiperparámetros de UMAP ─────────────────────────────────────────
if UMAP_DISPONIBLE:
    print("💡 GUÍA PARA ELEGIR HIPERPARÁMETROS DE UMAP")
    print("=" * 55)
    print()
    print("  n_neighbors (vecinos considerados):")
    print("    5–10   → Estructura muy local, muchos clusters pequeños")
    print("    15     → Balance recomendado (default=15)")
    print("    50–100 → Estructura global más prominente")
    print()
    print("  min_dist (separación mínima en el embedding):")
    print("    0.0    → Clusters muy compactos")
    print("    0.1    → Recomendado para exploración")
    print("    0.5    → Puntos más distribuidos, menos clusters")
    print()
    print("  metric para fingerprints:")
    print("    'jaccard'   → Tanimoto ← mejor para binarios")
    print("    'euclidean' → Menos apropiado")
    print()
    print("A diferencia de t-SNE, en UMAP las distancias globales")
    print("SÍ tienen cierto significado — clusters separados = ")
    print("familias químicas distintas.")


---
## 9. Análisis del espacio químico

Con las visualizaciones, podemos hacernos preguntas farmacológicamente relevantes:
- ¿Están los activos concentrados en regiones específicas?
- ¿Existen familias estructurales bien separadas?
- ¿El dataset cubre uniformemente el espacio, o está sesgado?


In [ ]:
# ── Análisis de cobertura: ¿los activos cubren el espacio? ──────────────────
# Usamos las coordenadas de t-SNE (o UMAP si está disponible)
coords_2d = X_umap if UMAP_DISPONIBLE else X_tsne
nombre_metodo = 'UMAP' if UMAP_DISPONIBLE else 't-SNE'

if 'activo' in df_fp_sub.columns:
    coords_activos   = coords_2d[df_fp_sub['activo'] == 1]
    coords_inactivos = coords_2d[df_fp_sub['activo'] == 0]

    print(f"ANÁLISIS DE COBERTURA DEL ESPACIO ({nombre_metodo})")
    print("=" * 55)
    print()
    print("Centroide de activos vs inactivos:")
    for nombre, coords in [('Activos', coords_activos), ('Inactivos', coords_inactivos)]:
        centroide = coords.mean(axis=0)
        dispersion = coords.std(axis=0).mean()
        print(f"  {nombre:<12}: centroide=({centroide[0]:.2f}, {centroide[1]:.2f})  "
              f"dispersión media={dispersion:.2f}")

    # Distancia entre centroides
    c_act = coords_activos.mean(axis=0)
    c_ina = coords_inactivos.mean(axis=0)
    dist_centroides = np.linalg.norm(c_act - c_ina)
    print(f"  Distancia entre centroides: {dist_centroides:.2f}")
    print()
    if dist_centroides > 5:
        print("✅ Buena separación entre activos e inactivos en el espacio.")
        print("   Sugiere que el modelo podrá aprender la frontera de decisión.")
    else:
        print("⚠️  Activos e inactivos se solapan en el espacio químico.")
        print("   El modelo puede tener dificultades para separar las clases.")


In [ ]:
# ── Mapa de densidad del espacio químico ────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, (clase, label, cmap, titulo) in zip(
    axes,
    [(1, 'Activos',  'Greens', f'Densidad de Activos ({nombre_metodo})'),
     (0, 'Inactivos','Reds',   f'Densidad de Inactivos ({nombre_metodo})')]):

    if 'activo' in df_fp_sub.columns:
        mask = df_fp_sub['activo'] == clase
        coords = coords_2d[mask]
    else:
        coords = coords_2d

    # KDE 2D
    from scipy.stats import gaussian_kde
    if len(coords) > 10:
        kde = gaussian_kde(coords.T)
        x_min, x_max = coords_2d[:, 0].min()-1, coords_2d[:, 0].max()+1
        y_min, y_max = coords_2d[:, 1].min()-1, coords_2d[:, 1].max()+1
        xx, yy = np.mgrid[x_min:x_max:100j, y_min:y_max:100j]
        zz = kde(np.vstack([xx.ravel(), yy.ravel()])).reshape(xx.shape)
        ax.contourf(xx, yy, zz, levels=15, cmap=cmap, alpha=0.8)
        ax.scatter(coords[:, 0], coords[:, 1], s=5,
                   alpha=0.3, c='black', edgecolors='none')
    ax.set_title(titulo, fontsize=11)
    ax.set_xlabel(f'{nombre_metodo} 1', fontsize=10)
    ax.set_ylabel(f'{nombre_metodo} 2', fontsize=10)
    ax.set_xticks([]); ax.set_yticks([])

plt.tight_layout()
plt.savefig('densidad_espacio_quimico.png', dpi=130, bbox_inches='tight')
plt.show()


---
## 10. Diversidad molecular y análisis de scaffolds

En drug discovery, queremos saber:
- ¿Cuántas **familias estructurales** distintas hay en el dataset?
- ¿Está el dataset sesgado hacia un solo scaffold?
- ¿Qué scaffolds contienen más activos?

El **scaffold de Murcko** es el núcleo estructural de una molécula  
(sin cadenas laterales), y se usa para agrupar moléculas por familia química.


In [ ]:
# ── Scaffolds de Murcko ──────────────────────────────────────────────────────
def get_murcko_scaffold(smiles, generico=False):
    """
    Extrae el scaffold de Murcko de una molécula.

    Parámetros
    ----------
    smiles   : str  — SMILES de la molécula
    generico : bool — si True, devuelve el scaffold genérico
                      (todos los átomos = carbono, todos los enlaces = simple)

    Retorna
    -------
    str — SMILES del scaffold, o None si falla
    """
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    try:
        scaffold = MurckoScaffold.GetScaffoldForMol(mol)
        if generico:
            scaffold = MurckoScaffold.MakeScaffoldGeneric(scaffold)
        return Chem.MolToSmiles(scaffold)
    except Exception:
        return None

# Calcular scaffolds para todo el dataset
print("Calculando scaffolds de Murcko...")
df_fp['scaffold']         = [get_murcko_scaffold(s) for s in tqdm(df_fp['std_smiles'])]
df_fp['scaffold_generico'] = [get_murcko_scaffold(s, generico=True) for s in df_fp['std_smiles']]

# Estadísticas de scaffolds
n_total     = len(df_fp)
n_scaffolds = df_fp['scaffold'].nunique()
n_unicos    = (df_fp['scaffold'].value_counts() == 1).sum()

print(f"\nANÁLISIS DE SCAFFOLDS")
print("=" * 45)
print(f"  Moléculas totales:          {n_total}")
print(f"  Scaffolds únicos:           {n_scaffolds}")
print(f"  Scaffolds singleton (n=1):  {n_unicos} ({n_unicos/n_scaffolds*100:.0f}%)")
print(f"  Ratio moléculas/scaffold:   {n_total/n_scaffolds:.2f}")


In [ ]:
# ── Top scaffolds y su actividad ────────────────────────────────────────────
scaffold_stats = df_fp.groupby('scaffold').agg(
    n_moleculas=('std_smiles', 'count'),
    pct_activos=('activo', 'mean') if 'activo' in df_fp.columns else ('std_smiles', 'count'),
    pActividad_media=('pActividad', 'mean') if 'pActividad' in df_fp.columns else ('std_smiles', 'count'),
).sort_values('n_moleculas', ascending=False)

print("TOP 10 SCAFFOLDS MÁS FRECUENTES")
print("=" * 65)
print(f"  {'Scaffold (primeros 45 chars)':<45} {'n':>5} {'%Activos':>10} {'pAct media':>12}")
print("  " + "-"*65)

for scaffold, row in scaffold_stats.head(10).iterrows():
    scaffold_str = str(scaffold)[:45] if scaffold else 'Sin scaffold'
    n = row['n_moleculas']
    pct = row.get('pct_activos', 0) * 100
    pact = row.get('pActividad_media', 0)
    print(f"  {scaffold_str:<45} {n:>5} {pct:>9.1f}% {pact:>11.2f}")


In [ ]:
# ── Clustering K-Means sobre fingerprints ───────────────────────────────────
# Agrupa las moléculas en familias estructurales automáticamente

N_CLUSTERS = 8   # número de familias a buscar

kmeans = KMeans(n_clusters=N_CLUSTERS, random_state=42, n_init=10)
clusters = kmeans.fit_predict(X_fp_sub)
df_fp_sub = df_fp_sub.copy()
df_fp_sub['cluster'] = clusters

# Visualizar clusters en el espacio 2D
paleta = plt.cm.get_cmap('tab10', N_CLUSTERS)
fig, ax = plt.subplots(figsize=(9, 7))

for k in range(N_CLUSTERS):
    mask_k = clusters == k
    n_k    = mask_k.sum()
    pct_act = df_fp_sub[mask_k]['activo'].mean() * 100 if 'activo' in df_fp_sub else 0
    ax.scatter(coords_2d[mask_k, 0], coords_2d[mask_k, 1],
               c=[paleta(k)], label=f'Cluster {k+1} (n={n_k}, {pct_act:.0f}% act.)',
               alpha=0.55, s=15, edgecolors='none')

ax.legend(fontsize=8, bbox_to_anchor=(1.01, 1), loc='upper left')
ax.set_title(f'K-Means (k={N_CLUSTERS}) sobre fingerprints Morgan
{nombre_metodo}',
             fontsize=12)
ax.set_xlabel(f'{nombre_metodo} 1', fontsize=11)
ax.set_ylabel(f'{nombre_metodo} 2', fontsize=11)
ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout()
plt.savefig('clusters_espacio_quimico.png', dpi=130, bbox_inches='tight')
plt.show()

# Resumen de clusters
print("\nRESUMEN DE CLUSTERS")
print("=" * 50)
for k in range(N_CLUSTERS):
    mask_k = clusters == k
    n_k   = mask_k.sum()
    pact  = df_fp_sub[mask_k]['pActividad'].mean() if 'pActividad' in df_fp_sub else 0
    pct_a = df_fp_sub[mask_k]['activo'].mean()*100 if 'activo' in df_fp_sub else 0
    print(f"  Cluster {k+1}: {n_k:>4} mol | pAct media={pact:.2f} | {pct_a:.0f}% activos")


In [ ]:
# ── Diversidad intra/inter cluster ──────────────────────────────────────────
# ¿Son los clusters internamente similares? ¿Se diferencian entre sí?

print("DIVERSIDAD MOLECULAR POR CLUSTER (Tanimoto medio interno)")
print("=" * 55)

for k in range(min(N_CLUSTERS, 5)):  # solo los primeros 5 para no tardar
    mask_k = clusters == k
    fps_k  = X_fp_sub[mask_k]
    n_k    = fps_k.shape[0]
    if n_k < 2:
        continue
    # Muestra de pares para calcular similitud media
    n_pares = min(200, n_k*(n_k-1)//2)
    idx_pares = np.random.choice(n_k, size=(n_pares, 2), replace=True)
    sims = [tanimoto(fps_k[a], fps_k[b]) for a, b in idx_pares if a != b]
    sim_media = np.mean(sims) if sims else 0
    barra = '█' * int(sim_media * 20)
    print(f"  Cluster {k+1}: T medio interno = {sim_media:.3f}  {barra}")

print()
print("💡 Cluster con T_interno alto → moléculas muy similares (familia homogénea)")
print("   Cluster con T_interno bajo → familia diversa o cluster mal definido")


---
## 11. Guardar la matriz de features para modelado

La **Semana 4** necesita tres archivos:
1. `features_descriptores_<target>.csv` — descriptores fisicoquímicos curados
2. `features_morgan_<target>.npy` — matriz numpy de fingerprints (más eficiente que CSV para binarios)
3. `labels_<target>.csv` — etiquetas y pActividad

Todos alineados por el mismo índice de moléculas.


In [ ]:
# ── Preparar la tabla final de features descriptores ────────────────────────
# Alinear todo al subconjunto con fingerprints válidos y descriptores completos

df_features = df_fp.reset_index(drop=True).copy()

# Añadir descriptores al DataFrame
for col in df_desc_final.columns:
    if col in df_desc_alineado.columns:
        df_features[col] = df_desc_alineado[col].values[:len(df_features)]

# Añadir coordenadas de reducción de dimensiones
df_features['pca_1'] = X_pca[:len(df_features), 0]
df_features['pca_2'] = X_pca[:len(df_features), 1]
df_features['tsne_1'] = X_tsne[:len(df_features), 0] if len(X_tsne) >= len(df_features) else np.nan
df_features['tsne_2'] = X_tsne[:len(df_features), 1] if len(X_tsne) >= len(df_features) else np.nan
if UMAP_DISPONIBLE:
    df_features['umap_1'] = X_umap[:len(df_features), 0] if len(X_umap) >= len(df_features) else np.nan
    df_features['umap_2'] = X_umap[:len(df_features), 1] if len(X_umap) >= len(df_features) else np.nan

print(f"Tabla de features: {df_features.shape[0]} moléculas × {df_features.shape[1]} columnas")
print()
print(df_features.head(3).to_string(index=False))


In [ ]:
# ── Guardar todos los archivos ───────────────────────────────────────────────
import os

TARGET_SLUG = TARGET_NAME.lower().replace(' ', '_').replace('/', '_')[:30]

# 1. Descriptores curados (CSV)
archivo_desc = f'features_descriptores_{TARGET_SLUG}.csv'
cols_desc = ['molecule_chembl_id', 'std_smiles'] + list(df_desc_final.columns) +             ['pca_1','pca_2','tsne_1','tsne_2']
if UMAP_DISPONIBLE:
    cols_desc += ['umap_1','umap_2']
cols_desc = [c for c in cols_desc if c in df_features.columns]
df_features[cols_desc].to_csv(archivo_desc, index=False)

# 2. Fingerprints Morgan (numpy — mucho más compacto que CSV)
archivo_fp = f'features_morgan_{TARGET_SLUG}.npy'
np.save(archivo_fp, X_fp)

# 3. Labels y metadata
archivo_labels = f'labels_{TARGET_SLUG}.csv'
cols_labels = ['molecule_chembl_id', 'std_smiles', 'standard_type',
               'IC50_nM', 'pActividad', 'activo', 'scaffold',
               'MW','MolLogP','QED']
cols_labels = [c for c in cols_labels if c in df_features.columns]
df_features[cols_labels].to_csv(archivo_labels, index=False)

print("✅ ARCHIVOS GUARDADOS")
print("=" * 50)
for archivo in [archivo_desc, archivo_fp, archivo_labels]:
    if os.path.exists(archivo):
        tam = os.path.getsize(archivo) / 1024
        print(f"  {archivo:<45} ({tam:.1f} KB)")

print()
print("Cómo cargar en NB-ML-01 (Semana 4):")
print()
print("  import pandas as pd, numpy as np")
print(f"  df     = pd.read_csv('{archivo_labels}')")
print(f"  X_desc = pd.read_csv('{archivo_desc}')")
print(f"  X_fp   = np.load('{archivo_fp}')")
print()
print("  y      = df['activo'].values       # clasificación")
print("  y_reg  = df['pActividad'].values   # regresión")


In [ ]:
# ── Resumen estadístico final del espacio químico ───────────────────────────
print("RESUMEN DEL ESPACIO QUÍMICO EXPLORADO")
print("=" * 55)
print()
print(f"  Target:                  {TARGET_NAME}")
print(f"  Moléculas en el dataset: {len(df_features)}")
print()
print("  Descriptores fisicoquímicos:")
print(f"    Iniciales (RDKit):     {len(DESCRIPTORES_SELECCIONADOS)}")
print(f"    Tras curación:         {df_desc_final.shape[1]}")
print()
print("  Fingerprints:")
print(f"    Morgan ECFP4:          {X_fp.shape[1]} bits")
print(f"    MACCS Keys:            {X_maccs.shape[1]} bits")
print()
print("  Espacio químico:")
print(f"    Scaffolds únicos:      {df_fp['scaffold'].nunique()}")
print(f"    Similitud T media:     {np.mean(similitudes):.3f}")
print(f"    Clusters (K-Means):    {N_CLUSTERS}")
print()
if 'activo' in df_features.columns:
    n_act = df_features['activo'].sum()
    n_ina = (df_features['activo']==0).sum()
    print(f"  Clases:")
    print(f"    Activos:               {n_act} ({n_act/len(df_features)*100:.1f}%)")
    print(f"    Inactivos:             {n_ina} ({n_ina/len(df_features)*100:.1f}%)")


---
## ✅ Resumen del notebook

| Sección | Técnica | Salida |
|---------|---------|--------|
| **2. Descriptores** | RDKit `MolecularDescriptorCalculator` | ~40 propiedades por molécula |
| **3. Fingerprints** | Morgan ECFP4, ECFP6, MACCS, RDKit FP | Vectores binarios de 166–2048 bits |
| **4. Similitud** | Tanimoto sobre Morgan FP | Distribución de similitudes del dataset |
| **5. Selección** | Varianza + correlación + NaN | Descriptores no redundantes para modelos |
| **6. PCA** | Reducción lineal | Loadings: qué descriptores dominan el espacio |
| **7. t-SNE** | Reducción no lineal local | Mapa 2D con estructura de clusters |
| **8. UMAP** | Reducción moderna | Mapa 2D más rápido y con estructura global |
| **9. Análisis** | KDE + centroides | ¿Están los activos en regiones distintas? |
| **10. Scaffolds** | Murcko + K-Means | Familias estructurales y diversidad |
| **11. Guardar** | CSV + .npy | Matriz lista para Semana 4 |

## 📅 Próximo notebook: NB-ML-01 — Modelos QSAR

Con los features y labels generados aquí, entrenarás:
- **Regresión** sobre `pActividad` (predicción continua)
- **Clasificación** sobre `activo` (Random Forest, SVM)
- Evaluación con AUC-ROC, F1, MCC

---
*NB-DATA-03 · Ciencia de Datos en Descubrimiento de Fármacos · UNAL 2026*
